In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/quality_report.csv
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_delivery_note_paper_ded96e100b14.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_product_label_closeup_d086a7d3b9a7.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_wallet_with_cards_4bb4c9721011.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_person_shopping_in_store_3c7efd19e955.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_phone_screen_text_cedf30b7186a.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg3_Vietnameese_store_e3e7a0cc7a0a.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_envelope_letter_763a68651d18.jpg
/kaggle/input/datasets/kenvippro/receipt-dataset/Receipt_dataset/Negative/neg_printed_text_on_white_paper_0811db5

KeyboardInterrupt: 

In [ ]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib

In [ ]:
import os
import shutil

# 1. Folders banana
base_dir = '/kaggle/working/training_data'
for cat in ['invoices', 'receipts', 'contracts']:
    os.makedirs(os.path.join(base_dir, cat), exist_ok=True)

# 2. Sare input folders scan karna
print("Scanning /kaggle/input for datasets...")
input_root = '/kaggle/input'

for root, dirs, files in os.walk(input_root):
    # Check karein ke folder ke naam mein hamari categories hain ya nahi
    folder_name = root.lower()
    
    target = None
    if 'invoice' in folder_name: target = 'invoices'
    elif 'lab-7' in folder_name or 'receipt' in folder_name: target = 'receipts'
    elif 'signature' in folder_name or 'contract' in folder_name: target = 'contracts'
    
    if target:
        # Sirf images copy karein (pehli 10)
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if image_files:
            for f in image_files[:10]:
                shutil.copy(os.path.join(root, f), os.path.join(base_dir, target))
            print(f"✅ Found {target} in: {root} (Copied {len(image_files[:10])} images)")

print("\n--- Summary ---")
for cat in ['invoices', 'receipts', 'contracts']:
    count = len(os.listdir(os.path.join(base_dir, cat)))
    print(f"Folder {cat}: {count} images ready.")

In [ ]:
import os
import shutil

# Target setup
target_dir = '/kaggle/working/training_data/receipts'
os.makedirs(target_dir, exist_ok=True)

# Searching for the hidden SROIE folder
found = False
print("Searching for SROIE images...")

for root, dirs, files in os.walk('/kaggle/input'):
    # Look for the SROIE folder specifically
    if 'sroie' in root.lower() and ('image' in root.lower() or 'img' in root.lower() or 'train' in root.lower()):
        images = [f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if images:
            for img in images[:20]: # Copy 20 samples
                shutil.copy(os.path.join(root, img), target_dir)
            print(f"✅ Success! Found receipts in: {root}")
            found = True
            break

if not found:
    print("❌ Still not found. Trying a direct path backup...")
    # Manual backup path check
    backup_path = '/kaggle/input/sroie-datasetv2/SROIE2019/train/img'
    if os.path.exists(backup_path):
        for img in os.listdir(backup_path)[:20]:
            shutil.copy(os.path.join(backup_path, img), target_dir)
        print("✅ Found via backup path!")
    else:
        print("❌ Final Attempt failed. Please verify the dataset name in the right-side 'Input' panel.")

print(f"\n--- New Summary ---")
print(f"Receipts: {len(os.listdir(target_dir))} images ready.")

In [ ]:
import pytesseract
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib
import os

documents, labels = [], []
base_dir = '/kaggle/working/training_data'
categories = ['invoices', 'receipts', 'contracts']

print("Starting OCR extraction. This may take a minute...")

for cat in categories:
    folder = os.path.join(base_dir, cat)
    print(f"Processing category: {cat}")
    for img_name in os.listdir(folder):
        try:
            img_path = os.path.join(folder, img_name)
            # Extract text using Tesseract
            text = pytesseract.image_to_string(Image.open(img_path))
            
            # Only add if the text is meaningful
            if len(text.strip()) > 10:
                documents.append(text)
                labels.append(cat)
        except Exception as e:
            continue

# Task 1.2: TF-IDF Vectorization and Logistic Regression
print("\nTraining the model...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X = vectorizer.fit_transform(documents)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X, labels)

# Task 1.3: Save Models
os.makedirs('models', exist_ok=True)
joblib.dump(vectorizer, 'models/vectorizer.pkl')
joblib.dump(classifier, 'models/classifier.pkl')

print(f"\n🚀 SUCCESS! Model trained on {len(documents)} documents.")
print("The following files are now in your /kaggle/working/models/ folder:")
print("- vectorizer.pkl")
print("- classifier.pkl")

In [ ]:
!pip install fastapi uvicorn python-multipart

In [ ]:
%%writefile app.py
from fastapi import FastAPI, UploadFile, File
import joblib
import pytesseract
from PIL import Image
import io
import re

app = FastAPI()

# Task 2.2: Load the saved models
vectorizer = joblib.load('models/vectorizer.pkl')
classifier = joblib.load('models/classifier.pkl')

@app.get("/")
def read_root():
    return {"message": "Intelligent Document Processing API is live!"}

@app.post("/classify")
async def classify_document(file: UploadFile = File(...)):
    # Read image and perform OCR
    contents = await file.read()
    image = Image.open(io.BytesIO(contents))
    text = pytesseract.image_to_string(image)
    
    # Task 2.1: Classification logic
    text_vectorized = vectorizer.transform([text])
    prediction = classifier.predict(text_vectorized)[0]
    
    # Task 2.3: Information Extraction (Regex)
    # Looking for dates (DD/MM/YYYY or YYYY-MM-DD)
    date_pattern = r'\d{2}[/-]\d{2}[/-]\d{4}|\d{4}-\d{2}-\d{2}'
    # Looking for Currency/Amounts (e.g., $100.00 or Total: 500)
    amount_pattern = r'\$?\d+\.\d{2}|Total[:\s]*\d+'
    
    dates = re.findall(date_pattern, text)
    amounts = re.findall(amount_pattern, text)
    
    return {
        "filename": file.filename,
        "document_type": prediction,
        "extracted_info": {
            "dates_found": dates,
            "amounts_found": amounts
        },
        "ocr_preview": text[:150].replace('\n', ' ') + "..."
    }

In [27]:
!pip install fastapi uvicorn pyngrok nest_asyncio python-multipart -q

from fastapi import FastAPI
import nest_asyncio
import uvicorn
from threading import Thread
from pyngrok import ngrok
import time

# Fix notebook async issue
nest_asyncio.apply()

# Create API
app = FastAPI()

@app.get("/")
def home():
    return {"message": "FastAPI Running Successfully"}

# Start FastAPI server
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server = Thread(target=run)
server.start()

# Wait for server startup
time.sleep(5)

# Add your ngrok token
ngrok.set_auth_token("3DWsQ28ABVzRFFtltTrUg0hHUh6_n4nUueJbNcUeu4Ky2zW3")

# Kill old tunnels
ngrok.kill()

# Create tunnel
public_url = ngrok.connect(8000)

print("OPEN THIS LINK:")
print(public_url)

print("\nSWAGGER UI:")
print(str(public_url) + "/docs")

INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


OPEN THIS LINK:
NgrokTunnel: "https://narrow-deftly-lather.ngrok-free.dev" -> "http://localhost:8000"

SWAGGER UI:
NgrokTunnel: "https://narrow-deftly-lather.ngrok-free.dev" -> "http://localhost:8000"/docs
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET / HTTP/1.1" 200 OK
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET /doc HTTP/1.1" 404 Not Found
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET /doc HTTP/1.1" 404 Not Found
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET / HTTP/1.1" 200 OK
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET /doc HTTP/1.1" 404 Not Found
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2407:d000:17:68b2:540b:f732:4e2d:9811:0 - "GET / HTTP/1.1" 200 OK
